# MOUSE CORE · Experiment 2 v1
## Lexikální rozmanitost, kontextová proměnlivost a významové vrstvení

Devět autorských pohádek tvoří jednu skupinu. **O kohoutkovi a slepičce** je samostatná lidová reference. Každý text zůstává celkem. Věty používáme jen jako lokální kontexty opakovaných slov, nikoli jako náhradu celého příběhu.

Experiment 2 přebírá celotextové vektory z Pilot v2, takže je znovu nepočítá. Nově zkoumá:
- lexikální rozmanitost a opakované slovní formy;
- věty, ve kterých se stejné slovo vrací;
- kontextovou proměnlivost těchto vět pomocí BGE-M3;
- polohu návratů v průběhu příběhu.

**Pozor:** odlišnost větných embeddingů není přímé měření různých významů. Může odrážet změnu tématu, postav nebo okolních slov. Dvanáct úhlů Semantic Gap je interpretační rámec, nikoli automaticky měřená veličina.

V Colabu zvol GPU, pokud je dostupné, a spusť Běh → Spustit vše. Na začátku nahraješ ZIP `MOUSE_CORE_v2_vysledky.zip`.

In [ ]:
!pip -q install -U FlagEmbedding pandas matplotlib

In [ ]:
import json, re, math, io, zipfile
from pathlib import Path
from collections import Counter, defaultdict
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from google.colab import files

REPO="https://raw.githubusercontent.com/artmisanovak-prog/generative_gen/main"
MODEL_NAME="BAAI/bge-m3"
OUT=Path("/content/MOUSE_CORE_Experiment2_v1"); OUT.mkdir(exist_ok=True)
ANGLES=["Analytický","Kulturní","Filozofický","Naivní","Jazykový","Metaforický","Asociační","Introspektivní","Ironický","Groteskní","Absurdní","Surrealistický"]
CARDS={
"mys":("Myš","doma/primarni_data/kriva_pohadka/mys.txt"),
"ryba":("Ryba","doma/primarni_data/kriva_pohadka/ryba.txt"),
"kocour":("Kocour","doma/primarni_data/kriva_pohadka/kocour.txt"),
"lemur":("Lemur","doma/primarni_data/kriva_pohadka/lemur.txt"),
"slon":("Slon","doma/primarni_data/kriva_pohadka/slon.txt"),
"pav":("Páv","doma/primarni_data/kriva_pohadka/pav.txt"),
"cap":("Čáp","doma/primarni_data/kriva_pohadka/cap.txt"),
"kozel":("Kozel","doma/primarni_data/kriva_pohadka/kozel.txt"),
"zajic":("Zajíc","doma/primarni_data/kriva_pohadka/zajic.txt")}
REF=("kohoutek_slepicka","O kohoutkovi a slepičce","doma/primarni_data/reference/lidova_pohadka_kohoutek_a_slepicka.txt")
print("Zařízení:", "GPU" if torch.cuda.is_available() else "CPU")

## 1 · Převzetí výsledků Pilot v2
Nahraj ZIP z předchozího běhu. Z něj převezmeme původní vektory celých textů a vektor lidové reference. Tyto vektory se nepřepočítávají.

In [ ]:
uploaded=files.upload()
zips=[n for n in uploaded if n.lower().endswith(".zip")]
assert zips, "Nahraj MOUSE_CORE_v2_vysledky.zip"
with zipfile.ZipFile(io.BytesIO(uploaded[zips[0]])) as z:
    names=z.namelist()
    vf=next((n for n in names if n.endswith("mouse_core_bge_m3_vectors_v2.json")),None)
    rf=next((n for n in names if n.endswith("mouse_core_bge_m3_reference_comparison_v2.json")),None)
    assert vf and rf, f"V ZIP chybí očekávané soubory: {names}"
    baseline_vectors=json.loads(z.read(vf))
    baseline_reference=json.loads(z.read(rf))
assert len(baseline_vectors)==9
print("Převzato 9 autorských celotextových vektorů a samostatná reference.")

## 2 · Načtení deseti celých textů
Z autorských souborů odstraníme pouze celé technické řádky začínající `Karta:` nebo `Text:`. Lidová pohádka se načítá beze změn.

In [ ]:
docs=[]
for cid,(name,path) in CARDS.items():
    raw=urlopen(f"{REPO}/{path}").read().decode("utf-8-sig")
    text=re.sub(r"(?m)^[ \\t]*(?:Karta|Text):[^\r\n]*(?:\r?\n|$)","",raw).strip()
    assert text, f"Prázdný text: {cid}"
    docs.append({"id":cid,"name":name,"group":"authored","source":path,"text":text})
rid,rname,rpath=REF
rt=urlopen(f"{REPO}/{rpath}").read().decode("utf-8-sig").strip()
assert rt.startswith("O kohoutkovi a slepičce")
docs.append({"id":rid,"name":rname,"group":"reference","source":rpath,"text":rt})
assert len(docs)==10
for d in docs: print(d["name"],len(d["text"]),"znaků",d["group"])

## 3 · Lexikální rozmanitost a kontexty opakování
Počítáme slovní formy po sjednocení velikosti písmen. Různé pády zůstávají různými formami. TTR je poměr různých forem ku všem výskytům, nikoli měřítko významové bohatosti. U každého opakovaného slova ukládáme celé věty, v nichž se objevuje, a jejich polohu v příběhu.

In [ ]:
WORD=re.compile(r"[^\W\d_]+(?:[’'-][^\W\d_]+)*",re.UNICODE)
SENT=re.compile(r"[^.!?…]+(?:[.!?…]+|$)",re.S)
def sentence_spans(t):
    out=[]
    for m in SENT.finditer(t):
        s=m.group().strip()
        if s:
            a=m.start()+len(m.group())-len(m.group().lstrip())
            out.append((a,a+len(s),s))
    return out
profiles=[]; occurrence_rows=[]
for d in docs:
    ms=list(WORD.finditer(d["text"])); toks=[m.group().casefold() for m in ms]; counts=Counter(toks)
    spans=sentence_spans(d["text"]); rep=defaultdict(list)
    for m in ms:
        form=m.group().casefold()
        if counts[form]>1:
            si=next((i for i,(a,b,s) in enumerate(spans) if a<=m.start()<b),None)
            if si is not None:
                a,b,s=spans[si]
                rep[form].append({"sentence":s,"sentence_index":si,"relative_position":m.start()/max(1,len(d["text"]))})
    repeated=[{"form":f,"count":len(v),"occurrences":v} for f,v in rep.items()]
    total=len(toks); unique=len(counts); hapax=sum(v==1 for v in counts.values())
    p={"id":d["id"],"name":d["name"],"group":d["group"],"characters":len(d["text"]),"word_tokens":total,"unique_forms":unique,"TTR":unique/total if total else 0,"hapax_count":hapax,"hapax_ratio":hapax/unique if unique else 0,"repeated_types":sum(v>1 for v in counts.values()),"repeated_token_share":sum(v for v in counts.values() if v>1)/total if total else 0,"sentence_count":len(spans),"top_words":counts.most_common(30),"repeated_forms":repeated}
    profiles.append(p)
    for item in repeated:
        for occ in item["occurrences"]:
            occurrence_rows.append({"doc_id":d["id"],"doc_name":d["name"],"group":d["group"],"form":item["form"],**occ})
profile_df=pd.DataFrame([{k:p[k] for k in ["name","group","word_tokens","unique_forms","TTR","hapax_ratio","repeated_types","repeated_token_share","sentence_count"]} for p in profiles])
display(profile_df)

## 4 · Kontextové embeddingy opakovaných slov
Každá věta obsahující opakovanou slovní formu dostane vlastní větný embedding BGE-M3. Potom porovnáme věty, v nichž se stejná forma vrací v rámci téže pohádky. Nejde o izolovaný embedding slova, ale o reprezentaci celé věty.

In [ ]:
from FlagEmbedding import BGEM3FlagModel
model=BGEM3FlagModel(MODEL_NAME,use_fp16=torch.cuda.is_available())
# Každou odlišnou větu v rámci příběhu zakódujeme jen jednou.
unique_sentences={}; 
for r in occurrence_rows: unique_sentences.setdefault((r["doc_id"],r["sentence"]),len(unique_sentences))
sentence_texts=[None]*len(unique_sentences)
for (doc_id,s),i in unique_sentences.items(): sentence_texts[i]=s
print("Větných kontextů:",len(sentence_texts))
vecs=[]
for start in range(0,len(sentence_texts),24):
    enc=model.encode(sentence_texts[start:start+24],batch_size=8,max_length=512,return_dense=True,return_sparse=False,return_colbert_vecs=False)
    vecs.extend(np.asarray(enc["dense_vecs"],dtype=np.float32))
vecs=np.asarray(vecs,dtype=np.float32)
for r in occurrence_rows: r["vector_index"]=unique_sentences[(r["doc_id"],r["sentence"])]
print("Rozměr:",vecs.shape)

## 5 · Kontextová proměnlivost
Pro každou opakovanou formu počítáme kosinovou podobnost všech dvojic jejích různých větných kontextů. Nižší průměrná podobnost znamená, že se větné reprezentace více liší. Neprokazuje to automaticky změnu významu slova.

In [ ]:
def cosine(a,b):
    den=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den else float("nan")
groups=defaultdict(list)
for r in occurrence_rows: groups[(r["doc_id"],r["form"])].append(r)
form_results=[]
for (did,form),rows in groups.items():
    distinct=[]; seen=set()
    for r in rows:
        if r["sentence"] not in seen: seen.add(r["sentence"]); distinct.append(r)
    if len(distinct)<2: continue
    sims=[cosine(vecs[distinct[i]["vector_index"]],vecs[distinct[j]["vector_index"]]) for i in range(len(distinct)) for j in range(i+1,len(distinct))]
    p=next(x for x in profiles if x["id"]==did)
    form_results.append({"doc_id":did,"doc_name":p["name"],"group":p["group"],"form":form,"occurrences":len(rows),"distinct_sentence_contexts":len(distinct),"pair_count":len(sims),"mean_context_similarity":float(np.mean(sims)),"mean_context_dissimilarity":float(1-np.mean(sims)),"contexts":[{"sentence":r["sentence"],"relative_position":r["relative_position"]} for r in distinct]})
form_df=pd.DataFrame(form_results)
display(form_df.sort_values("mean_context_dissimilarity",ascending=False).head(30))

## 6 · Srovnání textů a skupin
Souhrn je popisný. Texty s malým počtem opakovaných forem ve více různých větách budou mít méně podkladů pro kontextové srovnání. Průměr nepovažujeme za skóre kvality ani významové hloubky.

In [ ]:
summary=[]
for p in profiles:
    f=form_df[form_df.doc_id==p["id"]] if len(form_df) else pd.DataFrame()
    summary.append({"name":p["name"],"group":p["group"],"tokens":p["word_tokens"],"TTR":p["TTR"],"repeated_types":p["repeated_types"],"forms_with_multiple_contexts":len(f),"mean_context_dissimilarity":float(f.mean_context_dissimilarity.mean()) if len(f) else None})
summary_df=pd.DataFrame(summary); display(summary_df)
display(summary_df.groupby("group").mean(numeric_only=True))

## 7 · Mapa návratů v průběhu celého příběhu
Každý řádek je jedna opakovaná slovní forma. Body ukazují její výskyty od začátku do konce textu. Nejde o významové štítky.

In [ ]:
for p in profiles:
    rows=[(x["form"],o["relative_position"]) for x in p["repeated_forms"] for o in x["occurrences"]]
    if not rows: continue
    top=[x["form"] for x in sorted(p["repeated_forms"],key=lambda x:x["count"],reverse=True)[:18]]
    fig,ax=plt.subplots(figsize=(11,max(3.5,len(top)*.28)))
    for yi,form in enumerate(top):
        xs=[pos for f,pos in rows if f==form]; ax.scatter(xs,[yi]*len(xs),s=22)
    ax.set_yticks(range(len(top)),top); ax.set_xlim(0,1)
    ax.set_xlabel("Poloha v celém textu (0 = začátek, 1 = konec)")
    ax.set_title("Návraty opakovaných slov · "+p["name"]); ax.grid(axis="x",alpha=.2)
    plt.tight_layout(); plt.show()

## 8 · Dvanáct úhlů Semantic Gap
Úhly nejsou automaticky přiřazovány BGE-M3. Jedno místo může být relevantní z více úhlů současně. Zde se ukládají jako metodický rámec pro následnou interpretační anotaci.

In [ ]:
angle_framework={"principle":"Interpretační perspektivy, nikoli automatické třídy slov ani měřené dimenze.","multiple_angles_per_occurrence_allowed":True,"angles":[{"id":i+1,"name":a,"annotation_status":"not_annotated"} for i,a in enumerate(ANGLES)]}
print(json.dumps(angle_framework,ensure_ascii=False,indent=2))

## 9 · Export výsledků
Vzniknou samostatné soubory Experimentu 2. Původní ZIP a vektory Pilot v2 zůstávají nedotčené.

In [ ]:
profiles_payload={"experiment":"MOUSE CORE Experiment 2 v1","profiles":profiles,"angle_framework":angle_framework,"limits":["TTR počítá slovní formy, nikoli významy.","Větné embeddingy jsou kontextové zástupné ukazatele, ne přímé měření významu.","Odlišnost kontextů nedokazuje polysémii ani významový posun.","Celé pohádky zůstávají základními jednotkami analýzy."]}
vectors_payload={"model":MODEL_NAME,"description":"Vektory celých vět obsahujících opakované slovní formy; nejsou to samostatné vektory slov.","occurrences":occurrence_rows,"unique_sentence_vectors":[{"doc_id":did,"sentence":s,"vector":[float(x) for x in vecs[i]]} for (did,s),i in unique_sentences.items()]}
variability_payload={"model":MODEL_NAME,"measure":"průměrná párová kosinová odlišnost větných embeddingů stejné opakované slovní formy v jednom příběhu","results":form_results,"warning":"Kontextová proměnlivost není přímé měření různých lexikálních významů."}
(OUT/"mouse_core_experiment2_v1_profiles.json").write_text(json.dumps(profiles_payload,ensure_ascii=False,indent=2),encoding="utf-8")
(OUT/"mouse_core_experiment2_v1_context_vectors.json").write_text(json.dumps(vectors_payload,ensure_ascii=False),encoding="utf-8")
(OUT/"mouse_core_experiment2_v1_context_variability.json").write_text(json.dumps(variability_payload,ensure_ascii=False,indent=2),encoding="utf-8")
summary_df.to_csv(OUT/"mouse_core_experiment2_v1_summary.csv",index=False,encoding="utf-8-sig")
zip_path=OUT/"MOUSE_CORE_Experiment2_v1_vysledky.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file() and p!=zip_path: z.write(p,arcname=p.name)
print("Hotovo:",zip_path)
files.download(str(zip_path))